# UD6.03. Dónde se ejecuta la inferencia: el presupuesto y la factura

**Módulo 5073 · Programación de Inteligencia Artificial · Curso 2026/27**
Bloque 4 de los apuntes · Criterios **3.c** y **3.e**

---

Esta es la decisión de arquitectura más importante de la unidad, y la que más se toma por
inercia: *«lo mandamos a la nube y allí lo procesamos»*, o su contraria de moda, *«lo
hacemos todo en el borde, que es lo moderno»*. Las dos son consignas.

La forma correcta de decidirlo son **tres cuentas**, y este cuaderno hace las tres:

1. **El presupuesto de latencia.** ¿Cabe la decisión en el tiempo que tolera la aplicación?
2. **La factura.** ¿Cuánto cuesta cada arquitectura, y a partir de cuántos dispositivos se
   cruzan?
3. **La restricción dura.** ¿Hay algo que prohíba una de las dos, independientemente de lo
   que digan las dos cuentas anteriores?

El orden importa: la tercera manda sobre las otras dos.

> Las latencias de inferencia de este cuaderno salen medidas del cuaderno `UD6_02`. Si has
> ejecutado aquel en tu máquina, sustituye aquí tus números.


In [ ]:
import matplotlib.pyplot as plt
import numpy as np

# Medido en UD6_02 sobre un x86 de sobremesa. Si tu medición es otra, cámbiala aquí:
# todo el cuaderno se recalcula solo.
MS_INFERENCIA_DISPOSITIVO = 0.023      # TFLite float32, lote de 1
MS_INFERENCIA_SERVIDOR = 2.0           # servidor con lotes y más carga por petición

print(f"inferencia en el dispositivo: {MS_INFERENCIA_DISPOSITIVO} ms (medida en UD6_02)")


---

## 1. El presupuesto de latencia

Un presupuesto de latencia es una lista de tramos con su tiempo. Escribirlo entero es el
noventa por ciento del trabajo, porque obliga a nombrar tramos que de otro modo no se
cuentan: la cola del servicio, el camino de vuelta, la actuación.


In [ ]:
EN_EL_DISPOSITIVO = [
    ("lectura del sensor", 1.0),
    ("preprocesado", 0.5),
    ("inferencia en el dispositivo", MS_INFERENCIA_DISPOSITIVO),
    ("actuación", 2.0),
]

EN_LA_NUBE = [
    ("lectura del sensor", 1.0),
    ("preprocesado", 0.5),
    ("sensor -> pasarela (WiFi)", 3.0),
    ("pasarela -> nube (ida)", 25.0),
    ("cola y arranque del servicio", 15.0),
    ("inferencia en el servidor", MS_INFERENCIA_SERVIDOR),
    ("nube -> pasarela (vuelta)", 25.0),
    ("pasarela -> actuador", 3.0),
    ("actuación", 2.0),
]


def presupuesto(nombre, tramos):
    total = sum(ms for _, ms in tramos)
    print(nombre)
    for etiqueta, ms in tramos:
        barra = "#" * max(1, round(ms))
        print(f"    {etiqueta:<32}{ms:>7.2f} ms  {barra}")
    print(f"    {'TOTAL':<32}{total:>7.2f} ms")
    print()
    return total


t_dispositivo = presupuesto("EN EL DISPOSITIVO", EN_EL_DISPOSITIVO)
t_nube = presupuesto("EN LA NUBE", EN_LA_NUBE)

print(f"Relación: {t_nube / t_dispositivo:.0f} veces.")
print()
print("Fíjate en de dónde sale la diferencia: NO de la inferencia. La inferencia en el")
print(f"servidor es {MS_INFERENCIA_SERVIDOR / MS_INFERENCIA_DISPOSITIVO:.0f} veces más "
      f"lenta que en el dispositivo, pero son 2 ms sobre 76.")
print("Lo que domina son los dos viajes de red y la cola: 65 de los 76,5 ms, un 85 %.")
print("Un servidor más potente no arregla nada. Es un problema de distancia, no de CPU.")


### La tabla que decide

La relación de 22 veces no sirve para decidir nada por sí sola. Lo que decide es el
**presupuesto de la aplicación**: cuánto tiempo tolera antes de que la respuesta deje de
servir.


In [ ]:
CASOS = [
    ("Frenado de emergencia industrial", 10, "la máquina recorre 2 cm en 10 ms"),
    ("Detección de peatón a 50 km/h", 50, "el coche recorre 70 cm en 50 ms"),
    ("Palabra de activación de un altavoz", 200, "por encima se nota el retardo"),
    ("Alerta de cámara frigorífica", 60_000, "un minuto no cambia nada"),
    ("Informe de consumo mensual", 86_400_000, "un día no cambia nada"),
]

print(f"{'aplicación':<38}{'presupuesto':>15}{'dispositivo':>13}{'nube':>9}")
print("-" * 75)
for nombre, presu, _ in CASOS:
    print(f"{nombre:<38}{f'{presu:,} ms':>15}"
          f"{'cabe' if t_dispositivo <= presu else 'NO cabe':>13}"
          f"{'cabe' if t_nube <= presu else 'NO cabe':>9}")

print()
print("Por qué cada presupuesto es el que es:")
for nombre, _, razon in CASOS:
    print(f"  {nombre:<38}{razon}")

print()
print("La conclusión NO es «el dispositivo es mejor». Son dos conclusiones:")
print("  - En las dos primeras filas la nube no es una opción peor: no es una opción.")
print("  - En las tres últimas subir a la nube no cuesta nada de tiempo, y ahorra el")
print("    trabajo enorme de mantener modelos actualizados en mil aparatos.")
print()
print("Quien responda «depende» sin decir de qué, no ha contestado. Depende del")
print("presupuesto, y el presupuesto se escribe con un número y su razón.")


---

## 2. La factura: cuánto cuesta cada arquitectura

Segunda cuenta, y aquí hay que tener cuidado, porque la creencia extendida —*«el borde
ahorra dinero»*— **no se sostiene sola**. Depende de una variable, y no es la que casi todo
el mundo supone.

Las dos arquitecturas tienen estas estructuras de coste:

- **Todo a la nube**: nada de equipo, y un coste de enlace y de proceso proporcional a
  cuántos mensajes se envían.
- **Decidir en el dispositivo**: un equipo capaz de inferir, amortizado, y enlace **solo
  cuando hay algo que contar**.

Vamos a construirlas y a ver qué sale.


In [ ]:
EUR_POR_MILLON_INFERENCIAS = 0.20   # proceso en la nube, orden de magnitud de una
                                    # función gestionada con un modelo pequeño
BYTES_LECTURA = 44         # un PUBLISH de MQTT, medido en UD6_01
BYTES_EVENTO = 250         # un evento ya interpretado lleva más metadatos
EUR_EQUIPO = 35.0          # sobrecoste del aparato capaz de inferir, por dispositivo
AMORTIZACION_MESES = 36

HZ = 1.0
MENSAJES_MES = HZ * 86_400 * 30


def coste_nube(n, eur_por_mb, proporcion_eventos=None):
    '''Todo se sube y se decide arriba. El dispositivo no infiere.'''
    mb = n * MENSAJES_MES * BYTES_LECTURA / 1e6
    inferencias = n * MENSAJES_MES
    return mb * eur_por_mb + inferencias / 1e6 * EUR_POR_MILLON_INFERENCIAS


def coste_borde(n, eur_por_mb, proporcion_eventos=0.001):
    '''Se decide abajo y solo se publica lo que importa. La inferencia ya está pagada
    dentro del equipo, así que no aparece como coste variable.'''
    eventos = n * MENSAJES_MES * proporcion_eventos
    mb = eventos * BYTES_EVENTO / 1e6
    equipo = n * EUR_EQUIPO / AMORTIZACION_MESES
    return mb * eur_por_mb + eventos / 1e6 * EUR_POR_MILLON_INFERENCIAS + equipo


LINEA_FIJA = 0.09 / 1024        # 0,09 EUR/GB, la tarifa de salida de un gran proveedor

print(f"Tarifa de línea fija: {LINEA_FIJA:.6f} EUR/MB")
print()
print(f"{'dispositivos':>13}{'nube EUR/mes':>15}{'borde EUR/mes':>16}{'cuál gana':>12}")
print("-" * 56)
for n in [10, 100, 1_000, 10_000, 100_000]:
    cn, cb = coste_nube(n, LINEA_FIJA), coste_borde(n, LINEA_FIJA)
    print(f"{n:>13,}{cn:>15,.0f}{cb:>16,.0f}{'nube' if cn < cb else 'borde':>12}")

print()
print("La nube gana SIEMPRE, y por casi el doble. No hay ningún cruce.")
print("Con una conexión de línea fija barata, el borde NO se justifica por coste.")
print("Si has oído lo contrario, la afirmación venía sin esta cuenta detrás.")


### Entonces, ¿cuándo ahorra el borde?

Fíjate en que el número de dispositivos no ha decidido nada: las dos columnas crecen
proporcionalmente, así que la relación entre ellas es la misma con diez que con cien mil. La
escala no es la variable.

La variable es **lo que cuesta el enlace**. Y ahí el intervalo es enorme: mover un megabyte
por una línea fija cuesta una diezmilésima de euro; moverlo por una red móvil con una tarjeta
SIM por dispositivo cuesta miles de veces más. Un contador de agua enterrado en una arqueta
no tiene fibra.

Vamos a barrer la tarifa y a buscar el cruce.


In [ ]:
def cruce_en_tarifa(proporcion_eventos=0.001):
    '''Tarifa, en EUR/MB, a la que las dos arquitecturas cuestan lo mismo.

    Los dos costes son rectas en la tarifa, así que el cruce se despeja y no hace
    falta buscarlo. Devuelve None si no hay cruce con tarifa positiva, que pasa
    cuando el borde mueve MÁS bytes que la nube: entonces pierde a cualquier precio.'''
    mb_nube = MENSAJES_MES * BYTES_LECTURA / 1e6
    mb_borde = MENSAJES_MES * proporcion_eventos * BYTES_EVENTO / 1e6
    if mb_borde >= mb_nube:
        return None
    inferencias = MENSAJES_MES / 1e6 * EUR_POR_MILLON_INFERENCIAS
    fijo_borde = EUR_EQUIPO / AMORTIZACION_MESES
    diferencia_fija = (fijo_borde + proporcion_eventos * inferencias) - inferencias
    t = diferencia_fija / (mb_nube - mb_borde)
    return t if t > 0 else None


# Comprobación: a la tarifa del cruce las dos tienen que costar lo mismo.
t_cruce = cruce_en_tarifa()
assert abs(coste_nube(1_000, t_cruce) - coste_borde(1_000, t_cruce)) < 1e-6
print(f"Se cruzan a {t_cruce:.5f} EUR/MB, es decir, {t_cruce * 1024:.2f} EUR/GB.")
print("Por debajo de esa tarifa gana la nube; por encima, decidir en el dispositivo.")
print("(comprobado: a esa tarifa las dos cuestan lo mismo)")
print()

TARIFAS = [
    ("Salida de un gran proveedor de nube", 0.09 / 1024),
    ("Fibra empresarial, coste marginal", 0.001),
    ("Datos móviles, plan de empresa", 0.01),
    ("Tarjeta SIM de internet de las cosas", 0.50),
    ("Enlace satelital de baja velocidad", 5.00),
]
print(f"{'enlace':<40}{'EUR/MB':>10}{'nube':>12}{'borde':>10}{'gana':>8}")
print("-" * 80)
for nombre, tarifa in TARIFAS:
    cn, cb = coste_nube(1_000, tarifa), coste_borde(1_000, tarifa)
    print(f"{nombre:<40}{tarifa:>10.5f}{cn:>12,.0f}{cb:>10,.0f}"
          f"{'nube' if cn < cb else 'borde':>8}")
print()
print("(columnas de coste para 1.000 dispositivos, en EUR al mes)")
print()
print("Ahí está la respuesta de verdad, y no es la que se repite:")
print("  el borde no ahorra porque inferir abajo sea barato.")
print("  Ahorra cuando el ENLACE es caro, y entonces ahorra muchísimo.")


### Sensibilidad: qué supuesto manda

El cruce depende de cuatro supuestos, y el que más manda es la **proporción de eventos**:
cada cuánto el dispositivo tiene algo que contar. Decidir abajo solo ahorra si el dispositivo
**calla** la mayor parte del tiempo.

Y hay un límite duro que conviene ver antes de mirar la tabla. Una lectura cruda son 44 bytes
y un evento interpretado son 250, así que **a partir del momento en que el dispositivo habla
más de 44/250 de las veces, la arquitectura de eventos mueve más datos que la de subirlo
todo**, y entonces pierde a cualquier precio del enlace. Ese umbral es del 17,6 %, y no
depende de la tarifa ni de la escala: sale de los dos tamaños de mensaje.

Esto tiene una consecuencia de negocio que es justo la que pide el criterio 3.e: **el umbral
del modelo es una decisión económica**. Un detector mal ajustado que dispara el 30 % de las
veces no ahorra enlace, paga el equipo igual, y además inunda de alarmas a quien las atiende,
que es lo que enseñaba la UD4 sobre elegir el umbral según el coste.


In [ ]:
limite = BYTES_LECTURA / BYTES_EVENTO
print(f"Umbral por tamaño de mensaje: {limite:.1%} "
      f"({BYTES_LECTURA} bytes por lectura / {BYTES_EVENTO} por evento)")
print()
print(f"{'eventos':>10}{'tarifa de cruce (EUR/GB)':>28}{'lectura':>38}")
print("-" * 76)
for prop in [0.0001, 0.001, 0.01, 0.05, 0.1, 0.176, 0.3, 0.5]:
    t = cruce_en_tarifa(prop)
    if t is None:
        print(f"{prop:>9.2%}{'no hay cruce':>28}{'el borde mueve más bytes':>38}")
        continue
    eur_gb = t * 1024
    if eur_gb < 1:
        lectura = "el borde compensa casi siempre"
    elif eur_gb < 100:
        lectura = "compensa con enlace móvil"
    else:
        lectura = "solo con enlace carísimo"
    print(f"{prop:>9.2%}{eur_gb:>28,.1f}{lectura:>38}")

print()
print("Las dos últimas filas no son «sale caro»: son «no hay tarifa que lo arregle».")
print("Un dispositivo que habla un tercio del tiempo no está resumiendo nada.")
print()
print("El ahorro no viene de inferir en el borde. Viene de CALLAR.")
print("Inferir en el borde es lo que permite callar con criterio.")


In [ ]:
tarifas = np.logspace(-5, 1, 200)
fig, ejes = plt.subplots(1, 2, figsize=(11, 4))

ejes[0].plot(tarifas, [coste_nube(1_000, t) for t in tarifas],
             label="todo a la nube", color="#4477aa")
ejes[0].plot(tarifas, [coste_borde(1_000, t) for t in tarifas],
             label="decidir en el dispositivo", color="#cc6677")
ejes[0].axvline(t_cruce, color="0.4", ls="--", lw=1)
ejes[0].annotate(f"cruce: {t_cruce * 1024:.1f} EUR/GB",
                 xy=(t_cruce, coste_nube(1_000, t_cruce)),
                 xytext=(14, 26), textcoords="offset points", fontsize=9,
                 arrowprops=dict(arrowstyle="->", color="0.4", lw=0.8))
for nombre, tarifa in TARIFAS:
    ejes[0].plot([tarifa], [min(coste_nube(1_000, tarifa), coste_borde(1_000, tarifa))],
                 "o", ms=5, color="0.25")
ejes[0].set_xscale("log")
ejes[0].set_yscale("log")
ejes[0].set_xlabel("coste del enlace (EUR/MB)")
ejes[0].set_ylabel("EUR al mes, 1.000 dispositivos")
ejes[0].set_title("lo que decide es el precio del enlace")
ejes[0].legend(fontsize=8)

proporciones = np.logspace(-4, np.log10(limite * 0.97), 80)
ejes[1].plot(proporciones, [cruce_en_tarifa(p) * 1024 for p in proporciones],
             color="#117733")
ejes[1].axvline(limite, color="0.3", lw=1)
ejes[1].annotate(f"a partir del {limite:.1%}\nno hay cruce", xy=(limite, 10),
                 xytext=(-96, 0), textcoords="offset points", fontsize=8)
ejes[1].axhline(0.09, color="#4477aa", ls="--", lw=1,
                label="línea fija (0,09 EUR/GB)")
ejes[1].axhline(512, color="#cc6677", ls="--", lw=1,
                label="SIM de IoT (~512 EUR/GB)")
ejes[1].set_xscale("log")
ejes[1].set_yscale("log")
ejes[1].set_xlabel("proporción de momentos con evento")
ejes[1].set_ylabel("tarifa de cruce (EUR/GB)")
ejes[1].set_title("cuanto más habla el dispositivo, menos compensa")
ejes[1].legend(fontsize=8, loc="upper left")

fig.tight_layout()
plt.show()

baratos = sum(1 for _, t in TARIFAS if t < t_cruce)
print("En la figura de la izquierda, el borde solo gana a la derecha de la línea")
print(f"discontinua. Los puntos son las cinco tarifas de la tabla: {baratos} de las cinco")
print("caen a la izquierda, y esa es la razón de que tanta instalación suba todo a la")
print("nube y le salga bien.")


### El límite de este modelo, que hay que declarar

Este modelo tiene una propiedad que conviene ver y no esconder: **todos sus costes son
proporcionales al número de dispositivos**, así que la recomendación no depende de la escala.
Eso es cómodo y es una simplificación.

En la realidad hay costes fijos que rompen esa propiedad: la plataforma de gestión, el equipo
que mantiene los modelos desplegados, la certificación del aparato. Todos ellos van **en
contra del borde a escala pequeña**, porque mantener modelos actualizados en cuarenta
dispositivos cuesta casi lo mismo que en cuatro mil.

Si en PR6 recomiendas borde para una instalación de veinte aparatos, tienes que decir quién
va a mantener esos veinte modelos y cuánto cuesta.


---

## 3. La restricción dura

Las dos cuentas anteriores comparan opciones. Esta tercera **elimina** opciones, y por eso va
la última pero manda sobre las otras dos.

Si el dato es personal, dónde se procesa deja de ser una preferencia de arquitectura. Y hay
un matiz que se entiende mal muy a menudo: el Reglamento General de Protección de Datos no
prohíbe procesar en la nube. Lo que exige es base jurídica, minimización y garantías, y
**minimización** es la palabra que conecta con este cuaderno:

> Si el dispositivo decide en local y solo publica el resultado, **el dato en bruto nunca
> llega a existir fuera del aparato**.

Eso no es cumplir mejor: es tener menos que cumplir. Un dato que no se recoge no hay que
protegerlo, ni notificar su brecha, ni borrarlo cuando alguien lo pida.


In [ ]:
DECISIONES = [
    ("Contar personas en una tienda", "imagen de vídeo", "un número por minuto"),
    ("Detectar caídas en una residencia", "acelerómetro continuo", "un aviso de caída"),
    ("Clasificar piezas en una cinta", "imagen de la pieza", "buena / defectuosa"),
    ("Detectar arritmias en una pulsera", "electrocardiograma continuo", "un aviso"),
]

print(f"{'caso':<36}{'si sube todo':<30}{'si decide abajo':<24}")
print("-" * 90)
for caso, crudo, resultado in DECISIONES:
    print(f"{caso:<36}{crudo:<30}{resultado:<24}")

print()
print("La columna del medio es dato personal en tres de los cuatro casos, y en dos de")
print("ellos es dato de salud, que tiene protección reforzada. La columna de la derecha")
print("no es dato personal en ninguno.")
print()
print("Esa diferencia no se arregla cifrando mejor: se arregla NO recogiendo. Y la")
print("decisión de arquitectura de este cuaderno es exactamente lo que permite no")
print("recoger.")
print()
print("Ojo con el caso 3, el de las piezas: ahí no hay dato personal y la restricción")
print("dura no aplica. Aplicar protección de datos a un problema que no la tiene es")
print("tan mal análisis como no aplicarla donde toca.")


---

## 4. La plantilla de decisión

Cierra el cuaderno con lo que tienes que poder rellenar en PR6 para cualquier caso. Las tres
cuentas, en orden, y la respuesta.


In [ ]:
def decide(nombre, presupuesto_ms, n_dispositivos, eur_por_mb, dato_personal,
           proporcion_eventos):
    '''Aplica las tres cuentas del cuaderno, en orden, y razona la recomendación.'''
    cabe_nube = t_nube <= presupuesto_ms
    cn = coste_nube(n_dispositivos, eur_por_mb)
    cb = coste_borde(n_dispositivos, eur_por_mb, proporcion_eventos)

    print(nombre)
    print(f"  1. latencia    presupuesto {presupuesto_ms:,} ms   "
          f"nube {t_nube:.1f} ms -> {'cabe' if cabe_nube else 'NO cabe'}")
    print(f"  2. coste       enlace {eur_por_mb:.5f} EUR/MB   "
          f"nube {cn:,.0f}   borde {cb:,.0f} EUR/mes -> "
          f"{'nube' if cn < cb else 'borde'}")
    nota = "dato personal: minimizar" if dato_personal else "sin dato personal"
    print(f"  3. restricción {nota}")

    if not cabe_nube:
        respuesta, razon = "EN EL DISPOSITIVO", "la latencia no cabe: no hay elección"
    elif dato_personal:
        respuesta = "EN EL DISPOSITIVO"
        razon = "cabría en la nube, pero decidir abajo evita recoger el dato crudo"
    elif cb < cn:
        respuesta, razon = "EN EL DISPOSITIVO", "cabe en las dos y el enlace es caro"
    else:
        respuesta = "EN LA NUBE"
        razon = "cabe en las dos, no hay dato personal y el enlace es barato"
    print(f"  -> {respuesta}: {razon}")
    print()


decide("Frenado de emergencia en una prensa", 10, 40, 0.001, False, 0.001)
decide("Conteo de aforo en 12 tiendas", 60_000, 12, 0.001, True, 0.01)
decide("Control de calidad en cinta, 8 cámaras", 500, 8, 0.001, False, 0.02)
decide("Contadores de agua en arqueta, 20.000", 3_600_000, 20_000, 0.50, False, 0.001)
decide("Telemetría de 200 salas con fibra", 3_600_000, 200, LINEA_FIJA, False, 0.001)

print("Fíjate en los tres últimos, que es donde está la enseñanza:")
print("  - Las tiendas caben holgadamente en la nube y solo son 12, así que las dos")
print("    primeras cuentas dicen «nube». Manda la tercera.")
print("  - Los contadores de agua caben en la nube de sobra por tiempo, pero el")
print("    enlace es una SIM y ahí la segunda cuenta cambia de signo.")
print("  - Las salas con fibra no tienen ninguna razón para bajar al borde, y")
print("    recomendarlo sería seguir la moda en contra de los tres números.")
print()
print("Ese orden -latencia, coste, restricción- y esa disciplina de no recomendar")
print("nada que las tres cuentas no sostengan es lo que se evalúa en PR6.")


---

## Lo que hay que llevarse

1. **Escribe el presupuesto de latencia entero**, con los dos viajes de red y la cola. El 85 %
   del tiempo de la opción en la nube no es la inferencia: es la distancia.
2. **La relación entre arquitecturas no decide nada.** Decide el presupuesto de la
   aplicación, y hay que escribirlo con un número y su razón.
3. **Las dos arquitecturas tienen formas de coste distintas**, así que se cruzan. Encuentra
   el cruce y di con qué supuestos.
4. **El ahorro del borde viene de callar, no de inferir.** Si el dispositivo habla la mitad
   del tiempo, no ahorra nada.
5. **La restricción legal va la última y manda sobre las dos cuentas.** Y la manera técnica
   de cumplirla es no recoger el dato crudo.
6. **No apliques la restricción donde no toca.** Un sistema que cuenta piezas no tiene datos
   personales, y decir que sí es tan mal análisis como lo contrario.
